In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.utils import resample
from collections import defaultdict

In [2]:
# TCVファイルの読み込み
train_data = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_A" のテストデータだけを抽出
test_data = test[test["user_id"].str.endswith("_A")]

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:
        return 3  # 広告経由の購入
    elif row["event_type"] == 2:
        return 2  # 広告クリック
    elif row["event_type"] == 1:
        return 1  # 詳細ページ閲覧
    else:
        return 0  # カート追加

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

In [5]:
# タイムスタンプ処理
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])
train_data["time_since_last"] = (train_data["time_stamp"].max() - train_data["time_stamp"]).dt.total_seconds()

In [6]:
# ユーザーと商品の特徴量
user_features = train_data.groupby("user_id").agg(
    user_total_interactions=("product_id", "count"),
    user_unique_products=("product_id", "nunique"),
    user_avg_time_since=("time_since_last", "mean")
).reset_index()

product_features = train_data.groupby("product_id").agg(
    product_total_interactions=("user_id", "count"),
    product_unique_users=("user_id", "nunique")
).reset_index()

In [7]:
# **新しい特徴量（ユーザー・商品間の関係性）**
interaction_features = train_data.groupby(["user_id", "product_id"]).agg(
    user_product_interaction_count=("event_type", "count"),
    user_product_avg_time_since=("time_since_last", "mean")
).reset_index()

In [8]:
# データ結合
train_data = train_data.merge(user_features, on="user_id", how="left")
train_data = train_data.merge(product_features, on="product_id", how="left")
train_data = train_data.merge(interaction_features, on=["user_id", "product_id"], how="left")

In [9]:
# 学習用特徴量
features = [
    "user_total_interactions", "user_unique_products", "user_avg_time_since",
    "product_total_interactions", "product_unique_users",
    "user_product_interaction_count", "user_product_avg_time_since"
]

In [10]:
# データバランス調整（リサンプリング最適化）
train_data_pos = train_data[train_data["relevance"] >= 2]  
train_data_neg = train_data[train_data["relevance"] < 2]   

train_data_neg_undersampled = resample(train_data_neg, replace=True, n_samples=700000, random_state=42)
train_data_pos_oversampled = resample(train_data_pos, replace=True, n_samples=800000, random_state=42)

train_data_balanced = pd.concat([train_data_neg_undersampled, train_data_pos_oversampled])

In [11]:
# クロスバリデーション
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ndcg_scores = []

In [12]:
for train_index, val_index in kf.split(train_data_balanced):
    train_set = train_data_balanced.iloc[train_index]
    val_set = train_data_balanced.iloc[val_index]

    X_train, y_train = train_set[features], train_set["relevance"]
    X_val, y_val = val_set[features], val_set["relevance"]

    group_train = train_set.groupby("user_id").size().to_numpy()
    group_val = val_set.groupby("user_id").size().to_numpy()

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(group_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(group_val)

    params = {
        "objective": "rank:ndcg",
        "eval_metric": "ndcg",
        "booster": "gbtree",
        "eta": 0.05,  # 学習率を下げて汎化性能向上
        "max_depth": 7,
        "min_child_weight": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "lambda": 1.5,
        "gamma": 0.2
    }

    evals_result = {}
    model = xgb.train(params, dtrain, num_boost_round=500,
                      evals=[(dtrain, "train"), (dval, "val")],
                      evals_result=evals_result,
                      early_stopping_rounds=30,
                      verbose_eval=10)

    val_set_copy = val_set.copy()
    val_set_copy["score"] = model.predict(xgb.DMatrix(val_set_copy[features]))

    def ndcg_at_k(relevance_scores, k):
        relevance_scores = np.array(relevance_scores)
        ideal_relevance = np.sort(relevance_scores)[::-1]  # 降順ソート

        # kに満たない場合、ゼロ埋め
        if len(relevance_scores) < k:
            relevance_scores = np.pad(relevance_scores, (0, k - len(relevance_scores)), constant_values=0)
            ideal_relevance = np.pad(ideal_relevance, (0, k - len(ideal_relevance)), constant_values=0)

        dcg = np.sum(relevance_scores[:k] / np.log2(np.arange(2, k + 2)))
        ideal_dcg = np.sum(ideal_relevance[:k] / np.log2(np.arange(2, k + 2)))

        return dcg / ideal_dcg if ideal_dcg > 0 else 0

    ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()
    ndcg_scores.append(ndcg_val)

[0]	train-ndcg:0.94684	val-ndcg:0.96505
[10]	train-ndcg:0.95375	val-ndcg:0.96812
[20]	train-ndcg:0.95450	val-ndcg:0.96879
[30]	train-ndcg:0.95517	val-ndcg:0.96898
[40]	train-ndcg:0.95583	val-ndcg:0.96931
[50]	train-ndcg:0.95647	val-ndcg:0.96964
[60]	train-ndcg:0.95679	val-ndcg:0.96981
[70]	train-ndcg:0.95718	val-ndcg:0.96994
[80]	train-ndcg:0.95750	val-ndcg:0.97011
[90]	train-ndcg:0.95790	val-ndcg:0.97020
[100]	train-ndcg:0.95822	val-ndcg:0.97038
[110]	train-ndcg:0.95862	val-ndcg:0.97063
[120]	train-ndcg:0.95892	val-ndcg:0.97075
[130]	train-ndcg:0.95920	val-ndcg:0.97084
[140]	train-ndcg:0.95945	val-ndcg:0.97086
[150]	train-ndcg:0.95967	val-ndcg:0.97103
[160]	train-ndcg:0.95990	val-ndcg:0.97109
[170]	train-ndcg:0.96009	val-ndcg:0.97120
[180]	train-ndcg:0.96035	val-ndcg:0.97127
[190]	train-ndcg:0.96053	val-ndcg:0.97135
[200]	train-ndcg:0.96078	val-ndcg:0.97150
[210]	train-ndcg:0.96096	val-ndcg:0.97156
[220]	train-ndcg:0.96116	val-ndcg:0.97163
[230]	train-ndcg:0.96133	val-ndcg:0.97170
[24

/tmp/ipykernel_23428/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.94734	val-ndcg:0.96483
[10]	train-ndcg:0.95399	val-ndcg:0.96836
[20]	train-ndcg:0.95486	val-ndcg:0.96885
[30]	train-ndcg:0.95544	val-ndcg:0.96921
[40]	train-ndcg:0.95612	val-ndcg:0.96948
[50]	train-ndcg:0.95669	val-ndcg:0.96989
[60]	train-ndcg:0.95702	val-ndcg:0.97010
[70]	train-ndcg:0.95735	val-ndcg:0.97022
[80]	train-ndcg:0.95760	val-ndcg:0.97028
[90]	train-ndcg:0.95798	val-ndcg:0.97048
[100]	train-ndcg:0.95833	val-ndcg:0.97057
[110]	train-ndcg:0.95865	val-ndcg:0.97074
[120]	train-ndcg:0.95894	val-ndcg:0.97089
[130]	train-ndcg:0.95926	val-ndcg:0.97101
[140]	train-ndcg:0.95953	val-ndcg:0.97107
[150]	train-ndcg:0.95979	val-ndcg:0.97111
[160]	train-ndcg:0.96004	val-ndcg:0.97119
[170]	train-ndcg:0.96030	val-ndcg:0.97135
[180]	train-ndcg:0.96057	val-ndcg:0.97144
[190]	train-ndcg:0.96082	val-ndcg:0.97149
[200]	train-ndcg:0.96102	val-ndcg:0.97162
[210]	train-ndcg:0.96121	val-ndcg:0.97165
[220]	train-ndcg:0.96135	val-ndcg:0.97173
[230]	train-ndcg:0.96155	val-ndcg:0.97181
[24

/tmp/ipykernel_23428/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.94697	val-ndcg:0.96478
[10]	train-ndcg:0.95397	val-ndcg:0.96850
[20]	train-ndcg:0.95507	val-ndcg:0.96886
[30]	train-ndcg:0.95553	val-ndcg:0.96908
[40]	train-ndcg:0.95609	val-ndcg:0.96941
[50]	train-ndcg:0.95671	val-ndcg:0.96971
[60]	train-ndcg:0.95717	val-ndcg:0.97001
[70]	train-ndcg:0.95750	val-ndcg:0.97010
[80]	train-ndcg:0.95785	val-ndcg:0.97021
[90]	train-ndcg:0.95819	val-ndcg:0.97034
[100]	train-ndcg:0.95856	val-ndcg:0.97051
[110]	train-ndcg:0.95883	val-ndcg:0.97065
[120]	train-ndcg:0.95907	val-ndcg:0.97073
[130]	train-ndcg:0.95937	val-ndcg:0.97092
[140]	train-ndcg:0.95958	val-ndcg:0.97101
[150]	train-ndcg:0.95984	val-ndcg:0.97117
[160]	train-ndcg:0.96004	val-ndcg:0.97129
[170]	train-ndcg:0.96025	val-ndcg:0.97141
[180]	train-ndcg:0.96052	val-ndcg:0.97151
[190]	train-ndcg:0.96074	val-ndcg:0.97156
[200]	train-ndcg:0.96100	val-ndcg:0.97166
[210]	train-ndcg:0.96115	val-ndcg:0.97174
[220]	train-ndcg:0.96132	val-ndcg:0.97180
[230]	train-ndcg:0.96142	val-ndcg:0.97188
[24

/tmp/ipykernel_23428/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.94777	val-ndcg:0.96459
[10]	train-ndcg:0.95411	val-ndcg:0.96788
[20]	train-ndcg:0.95502	val-ndcg:0.96847
[30]	train-ndcg:0.95575	val-ndcg:0.96884
[40]	train-ndcg:0.95645	val-ndcg:0.96929
[50]	train-ndcg:0.95707	val-ndcg:0.96949
[60]	train-ndcg:0.95745	val-ndcg:0.96969
[70]	train-ndcg:0.95777	val-ndcg:0.96986
[80]	train-ndcg:0.95807	val-ndcg:0.96999
[90]	train-ndcg:0.95841	val-ndcg:0.97013
[100]	train-ndcg:0.95871	val-ndcg:0.97032
[110]	train-ndcg:0.95896	val-ndcg:0.97048
[120]	train-ndcg:0.95921	val-ndcg:0.97062
[130]	train-ndcg:0.95957	val-ndcg:0.97074
[140]	train-ndcg:0.95974	val-ndcg:0.97086
[150]	train-ndcg:0.95994	val-ndcg:0.97090
[160]	train-ndcg:0.96010	val-ndcg:0.97099
[170]	train-ndcg:0.96031	val-ndcg:0.97107
[180]	train-ndcg:0.96056	val-ndcg:0.97117
[190]	train-ndcg:0.96083	val-ndcg:0.97126
[200]	train-ndcg:0.96100	val-ndcg:0.97134
[210]	train-ndcg:0.96117	val-ndcg:0.97143
[220]	train-ndcg:0.96133	val-ndcg:0.97151
[230]	train-ndcg:0.96147	val-ndcg:0.97155
[24

/tmp/ipykernel_23428/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.94709	val-ndcg:0.96513
[10]	train-ndcg:0.95337	val-ndcg:0.96836
[20]	train-ndcg:0.95438	val-ndcg:0.96901
[30]	train-ndcg:0.95487	val-ndcg:0.96919
[40]	train-ndcg:0.95564	val-ndcg:0.96957
[50]	train-ndcg:0.95617	val-ndcg:0.96983
[60]	train-ndcg:0.95678	val-ndcg:0.96997
[70]	train-ndcg:0.95717	val-ndcg:0.97017
[80]	train-ndcg:0.95739	val-ndcg:0.97042
[90]	train-ndcg:0.95775	val-ndcg:0.97066
[100]	train-ndcg:0.95809	val-ndcg:0.97078
[110]	train-ndcg:0.95847	val-ndcg:0.97094
[120]	train-ndcg:0.95870	val-ndcg:0.97104
[130]	train-ndcg:0.95894	val-ndcg:0.97122
[140]	train-ndcg:0.95921	val-ndcg:0.97130
[150]	train-ndcg:0.95951	val-ndcg:0.97138
[160]	train-ndcg:0.95983	val-ndcg:0.97148
[170]	train-ndcg:0.96006	val-ndcg:0.97160
[180]	train-ndcg:0.96032	val-ndcg:0.97170
[190]	train-ndcg:0.96055	val-ndcg:0.97179
[200]	train-ndcg:0.96074	val-ndcg:0.97191
[210]	train-ndcg:0.96086	val-ndcg:0.97195
[220]	train-ndcg:0.96107	val-ndcg:0.97207
[230]	train-ndcg:0.96128	val-ndcg:0.97212
[24

/tmp/ipykernel_23428/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


In [13]:
print(f"Average nDCG@22 from CV: {np.mean(ndcg_scores):.4f}")

Average nDCG@22 from CV: 0.7528


In [14]:
# テストデータの処理
candidate_products = train_data.groupby("product_id")["user_id"].count().reset_index()
candidate_products = candidate_products.sort_values("user_id", ascending=False).head(22)  # 上位22商品

In [15]:
# すべてのユーザーに候補商品を紐づける
test_data = test_data.assign(key=1).merge(candidate_products.assign(key=1), on="key").drop("key", axis=1)
test_data = test_data.rename(columns={'user_id_x': 'user_id'}).drop(columns=['user_id_y'])

In [16]:
# 評価データの前処理（学習データと同じ特徴量処理を適用）
test_data = test_data.merge(user_features, on="user_id", how="left")
test_data = test_data.merge(product_features, on="product_id", how="left")
test_data = test_data.merge(interaction_features, on=["user_id", "product_id"], how="left")
test_data.fillna(0, inplace=True)

In [17]:
# 予測用データ
X_test = test_data[features]
dtest = xgb.DMatrix(X_test)

In [18]:
# 予測スコアの計算
test_data["score"] = model.predict(dtest)

In [19]:
# 各ユーザーごとにランキング付け（スコアが高い順）
test_data["rank"] = test_data.groupby("user_id")["score"].rank(ascending=False, method="first")

In [20]:
# テストデータの nDCG 計算
ground_truth_test = defaultdict(dict)
for _, row in train_data.iterrows():
    ground_truth_test[row["user_id"]][row["product_id"]] = row["relevance"]

def evaluate_ndcg(data, ground_truth, k=22):
    ndcg_scores = []
    for user_id, group in data.groupby("user_id"):
        predicted_items = group.sort_values("score", ascending=False)["product_id"].tolist()
        relevance_scores = [ground_truth[user_id].get(item, 0) for item in predicted_items]
        ndcg_scores.append(ndcg_at_k(relevance_scores, k))
    return np.mean(ndcg_scores) if ndcg_scores else 0

ndcg_test = evaluate_ndcg(test_data, ground_truth_test, k=22)
print(f"Test nDCG@22: {ndcg_test:.4f}")

Test nDCG@22: 0.2601


In [21]:
# 提出用データの作成（上位 22 件のみ）
submission = test_data[test_data["rank"] <= 22][["user_id", "product_id", "rank"]]
submission['rank'] = submission['rank'].astype(int)

In [22]:
# 保存
submission.to_csv('./predict_file/predictions_A.tsv', sep="\t", index=False)